## Day 2: Realized Volatility, Features, and Time Splits

This notebook implements:
1) 10-day realized volatility target,
2) feature engineering,
3) train/validation/test preparation,
4) quick EDA plots for AAPL and MSFT.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name.startswith("notebooks") else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data:"
if not DATA_DIR.exists():
    DATA_DIR = PROJECT_ROOT / "data"

FIG_DIR = PROJECT_ROOT / "figures:"
if not FIG_DIR.exists():
    FIG_DIR = PROJECT_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Using data dir:", DATA_DIR)
print("Using figures dir:", FIG_DIR)


aapl = pd.read_csv(DATA_DIR / "AAPL_clean.csv", parse_dates=["Date"])
msft = pd.read_csv(DATA_DIR / "MSFT_clean.csv", parse_dates=["Date"])

# Normalize to tz-naive timestamps for clean date filtering.
aapl["Date"] = pd.to_datetime(aapl["Date"], utc=True).dt.tz_localize(None)
msft["Date"] = pd.to_datetime(msft["Date"], utc=True).dt.tz_localize(None)

print("AAPL rows:", len(aapl), "MSFT rows:", len(msft))

Using data dir: /Users/syedm/Desktop/MSAI/CSML/CSML FInal Project/data:
Using figures dir: /Users/syedm/Desktop/MSAI/CSML/CSML FInal Project/figures:
AAPL rows: 1256 MSFT rows: 1256


In [2]:
def add_volatility(df, window=10):
    df = df.sort_values("Date").copy()
    df["log_return"] = np.log(df["Close"]).diff()
    df["rv_10"] = df["log_return"].rolling(window).std()
    return df


def make_target(df):
    df = df.sort_values("Date").copy()
    df["target_rv_10"] = df["rv_10"].shift(-1)
    return df


aapl = make_target(add_volatility(aapl, window=10))
msft = make_target(add_volatility(msft, window=10))

for df_ in (aapl, msft):
    df_.dropna(subset=["log_return", "rv_10", "target_rv_10"], inplace=True)

print("After target prep -> AAPL:", len(aapl), "MSFT:", len(msft))

After target prep -> AAPL: 1245 MSFT: 1245


In [3]:
def add_features(df, n_return_lags=5):
    df = df.sort_values("Date").copy()

    df["log_close"] = np.log(df["Close"])

    for k in range(1, n_return_lags + 1):
        df[f"log_return_lag_{k}"] = df["log_return"].shift(k)

    df["rv_10_lag_1"] = df["rv_10"].shift(1)

    df["log_volume"] = np.log(df["Volume"].replace(0, np.nan))
    df["volume_change"] = df["Volume"].pct_change()

    return df


aapl = add_features(aapl)
msft = add_features(msft)

for df_ in (aapl, msft):
    df_.dropna(inplace=True)

print("After features/dropna -> AAPL:", len(aapl), "MSFT:", len(msft))
aapl.head()

After features/dropna -> AAPL: 1240 MSFT: 1240


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Company,log_return,...,target_rv_10,log_close,log_return_lag_1,log_return_lag_2,log_return_lag_3,log_return_lag_4,log_return_lag_5,rv_10_lag_1,log_volume,volume_change
15,2018-12-26 05:00:00,35.584984,37.727760,35.205859,37.713364,234330000,0.0,0.0,AAPL,0.068052,...,0.032296,3.630015,-0.026214,-0.039672,-0.025559,-0.031688,0.012909,0.019180,19.272241,0.576103
16,2018-12-27 05:00:00,37.394243,37.617401,36.009718,37.468628,212468400,0.0,0.0,AAPL,-0.006511,...,0.031786,3.623504,0.068052,-0.026214,-0.039672,-0.025559,-0.031688,0.032476,19.174304,-0.093294
17,2018-12-28 05:00:00,37.792565,38.037317,37.084705,37.487823,169165600,0.0,0.0,AAPL,0.000512,...,0.031108,3.624016,-0.006511,0.068052,-0.026214,-0.039672,-0.025559,0.032296,18.946389,-0.203808
18,2018-12-31 05:00:00,38.039701,38.238862,37.547797,37.850140,140014000,0.0,0.0,AAPL,0.009618,...,0.031114,3.633635,0.000512,-0.006511,0.068052,-0.026214,-0.039672,0.031786,18.757253,-0.172326
19,2019-01-02 05:00:00,37.166277,38.116491,37.007907,37.893333,148158800,0.0,0.0,AAPL,0.001141,...,0.043824,3.634775,0.009618,0.000512,-0.006511,0.068052,-0.026214,0.031108,18.813795,0.058171


In [4]:
def time_split(df, train_end, val_end):
    train = df[df["Date"] <= train_end].copy()
    val = df[(df["Date"] > train_end) & (df["Date"] <= val_end)].copy()
    test = df[df["Date"] > val_end].copy()
    return train, val, test


train_end = pd.Timestamp("2021-12-31")
val_end = pd.Timestamp("2022-12-31")

aapl_train, aapl_val, aapl_test = time_split(aapl, train_end, val_end)
msft_train, msft_val, msft_test = time_split(msft, train_end, val_end)

feature_cols = [
    c
    for c in aapl.columns
    if c.startswith("log_return_lag_")
    or c in ["log_close", "rv_10", "rv_10_lag_1", "log_volume", "volume_change"]
]
target_col = "target_rv_10"

print("Feature columns:", feature_cols)
print("AAPL split sizes:", len(aapl_train), len(aapl_val), len(aapl_test))
print("MSFT split sizes:", len(msft_train), len(msft_val), len(msft_test))

Feature columns: ['rv_10', 'log_close', 'log_return_lag_1', 'log_return_lag_2', 'log_return_lag_3', 'log_return_lag_4', 'log_return_lag_5', 'rv_10_lag_1', 'log_volume', 'volume_change']
AAPL split sizes: 760 252 228
MSFT split sizes: 760 252 228


In [5]:
PROCESSED_DIR = DATA_DIR / "processed_day2"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def save_split(prefix, train_df, val_df, test_df):
    split_map = {
        "train": train_df,
        "val": val_df,
        "test": test_df,
    }

    for split_name, split_df in split_map.items():
        x_path = PROCESSED_DIR / f"X_{prefix}_{split_name}.csv"
        y_path = PROCESSED_DIR / f"y_{prefix}_{split_name}.csv"
        split_df[feature_cols].to_csv(x_path, index=False)
        split_df[["Date", target_col]].to_csv(y_path, index=False)


save_split("AAPL", aapl_train, aapl_val, aapl_test)
save_split("MSFT", msft_train, msft_val, msft_test)

print("Saved processed files to:", PROCESSED_DIR)
sorted([p.name for p in PROCESSED_DIR.glob("*.csv")])[:6]

Saved processed files to: /Users/syedm/Desktop/MSAI/CSML/CSML FInal Project/data:/processed_day2


['X_AAPL_test.csv',
 'X_AAPL_train.csv',
 'X_AAPL_val.csv',
 'X_MSFT_test.csv',
 'X_MSFT_train.csv',
 'X_MSFT_val.csv']

In [6]:
for name, df_ in [("AAPL", aapl), ("MSFT", msft)]:
    plt.figure(figsize=(8, 4))
    plt.plot(df_["Date"], df_["Close"])
    plt.title(f"{name} Close Price")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}_price.png", dpi=150)
    plt.close()

    plt.figure(figsize=(8, 4))
    plt.plot(df_["Date"], df_["rv_10"])
    plt.title(f"{name} 10-day Realized Volatility")
    plt.xlabel("Date")
    plt.ylabel("RV_10")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}_rv10.png", dpi=150)
    plt.close()

print("Saved figures:")
for p in sorted(FIG_DIR.glob("AAPL_*.png")) + sorted(FIG_DIR.glob("MSFT_*.png")):
    print(" -", p.name)

Saved figures:
 - AAPL_price.png
 - AAPL_rv10.png
 - MSFT_price.png
 - MSFT_rv10.png


### Day 2 Checklist (Implemented)

- Target definition: 10-day realized volatility (`rv_10`) and 1-day-ahead target (`target_rv_10`).
- Features: return lags, volatility lag, log price, log volume, volume change.
- Splits: time-based train/val/test (2019-2021, 2022, 2023).
- Outputs:
  - Processed feature/target CSVs in `data:/processed_day2` (or `data/processed_day2`).
  - EDA figures in `figures:` (or `figures/`).